# Fase 01 — Entendimento dos dados (data_understanding)

**Pergunta:** o que os dados representam e estão prontos para uso?

**Hipótese:** as variáveis físicas do imóvel + demografia do zipcode são suficientes para descrever a
formação de preço nesse mercado, uma vez limpas as inconsistências pontuais.

**Módulo:** `src/scripts/build_dataset.py::run_dataset_build_pipeline` (usa `src/data/clean.py`,
`src/data/merge.py`).

## Dicionário de variáveis (principais)

**Físicas do imóvel (kc_house_data.csv):** `price` (alvo), `bedrooms`/`bathrooms` (contagem de
cômodos), `sqft_living`/`sqft_lot` (área construída / do terreno, em pés²), `floors`, `waterfront`
(binário, frente d'água), `view` (0-4, qualidade da vista), `condition` (1-5, estado de conservação),
`grade` (1-13, qualidade de construção/design — escala Seattle), `sqft_above`/`sqft_basement`
(área acima/abaixo do solo), `yr_built`/`yr_renovated`, `zipcode`/`lat`/`long` (localização — zipcode
só serve de chave de merge/agrupamento, nunca entra como feature categórica no modelo, ver
`roadmap/PRINCIPLES.md` P1), `sqft_living15`/`sqft_lot15` (área média dos 15 vizinhos mais próximos —
já é uma feature contextual pré-calculada pelo fornecedor dos dados).

**Demográficas do zipcode (zipcode_demographics.csv):** população total/urbana/suburbana/rural,
renda média domiciliar/per capita, valor médio de imóveis (`hous_val_amt`), distribuição de nível
educacional (%).

## Decisões de limpeza (independentes, verificadas nos dados reais — não copiadas do projeto anterior)

1. **1 linha removida** (`id=2402100895`): `bedrooms=33` com `sqft_living=1620` (~49 sqft/quarto) —
   fisicamente implausível para configuração residencial real, erro de digitação (não é outlier de
   mercado real, que seria fisicamente consistente mesmo sendo extremo).
2. **353 linhas mantidas** representando 177 imóveis revendidos dentro do período do dataset (mesmo
   `id`, `date`/`price` diferentes) — tratadas como observações de mercado independentes (cada venda é
   um evento de precificação real), não como duplicatas a remover. Risco de leakage entre train/test
   é baixo porque o split canônico (fase 02) é por `zipcode`, então revendas do mesmo imóvel sempre
   caem no mesmo lado do split — mas será checado explicitamente na fase 02.
3. **`price_log = log1p(price)`**: skewness bruto de `price` ≈ 4.02, cai para ≈ 0.43 em `log1p` —
   verificado empiricamente nos dados reais (ver gráfico abaixo), não herdado do blueprint sem checagem.
4. **Merge com demografia**: 70 zipcodes em ambos os arquivos, cobertura 1:1 completa (sem zipcode
   órfão em nenhum dos lados) — merge seguro.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.build_dataset import run_dataset_build_pipeline

report = run_dataset_build_pipeline()
report

{'dropped_implausible_bedrooms_ids': [2402100895],
 'n_rows_out': 21612,
 'n_resales': 353,
 'n_rows_in': 21613,
 'n_columns_out': 48}

In [3]:
import pandas as pd

df = pd.read_parquet("data/processed/house_clean.parquet")
print("shape:", df.shape)
df.head()

shape: (21612, 48)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,per_sbrbn,per_farm,per_non_farm,per_less_than_9,per_9_to_12,per_hsd,per_some_clg,per_assoc,per_bchlr,per_prfsnl
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,0.0,0.0,0.0,4.0,8.0,20.0,21.0,5.0,12.0,4.0
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,0.0,0.0,0.0,2.0,5.0,15.0,20.0,5.0,20.0,12.0
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,0.0,0.0,0.0,1.0,4.0,13.0,20.0,6.0,19.0,9.0
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,0.0,0.0,0.0,1.0,3.0,13.0,20.0,7.0,25.0,12.0
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,0.0,0.0,0.0,1.0,5.0,15.0,19.0,5.0,19.0,7.5


In [4]:
df[["price", "price_log", "bedrooms", "bathrooms", "sqft_living", "grade"]].describe()

,price,price_log,bedrooms,bathrooms,sqft_living,grade
count,2.161200e+04,21612.000000,21612.000000,21612.000000,21612.000000,21612.000000
mean,5.400835e+05,13.047804,3.369471,2.114774,2079.921016,7.656904
std,3.671351e+05,0.526691,0.907982,0.770177,918.456818,1.175477
min,7.500000e+04,11.225257,0.000000,0.000000,290.000000,1.000000
25%,3.218375e+05,12.681805,3.000000,1.750000,1426.500000,7.000000
50%,4.500000e+05,13.017005,3.000000,2.250000,1910.000000,7.000000
75%,6.450000e+05,13.377007,4.000000,2.500000,2550.000000,8.000000
max,7.700000e+06,15.856731,11.000000,8.000000,13540.000000,13.000000


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["price"], bins=60, color="#4C72B0")
axes[0].set_title(f"price (skew={df['price'].skew():.2f})")
axes[1].hist(df["price_log"], bins=60, color="#55A868")
axes[1].set_title(f"price_log = log1p(price) (skew={df['price_log'].skew():.2f})")
plt.tight_layout()
plt.savefig("reports/figures/01_price_log_transform.png", dpi=110)
plt.close()
print("saved reports/figures/01_price_log_transform.png")

saved reports/figures/01_price_log_transform.png


## Resultado (execução real acima)

`data/processed/house_clean.parquet` com 21.612 linhas × 48 colunas (`n_rows_in=21613` → 1 linha
implausível removida, `id=2402100895`). `n_resales=353` linhas de revenda mantidas como observações
independentes. `price_log` confirmado como alvo: skew cai de ≈4.02 (`price`) para ≈0.43 (`price_log`),
ver histogramas acima (números re-verificados na execução do `report` acima, não copiados).

## Decisão

`data/processed/house_clean.parquet` pronto para split canônico (fase 02). `price_log` confirmado
como alvo de modelagem.

## Artefatos

`data/processed/house_clean.parquet`, `reports/build_dataset_log.json`,
`reports/figures/01_price_log_transform.png`